In [ ]:
"""
LOSO (general) training  +  FEATURE METHOD 2: concatenate-after-encoder.

Leave-one-subject-out across all three configs. Two scalar gait features
(cadence, duration variability) enter through a SECOND input branch and are
concatenated onto the encoder output before the final dense layer (functional API).

Set USE_FEATURES = False for the sequence-only baseline (single input).

Leak-safety: duration variability is a per-person trait computed over each
subject's own cycles; cadence is per-cycle; scalars standardized on each fold's
training subjects only.
"""
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from scipy.interpolate import interp1d
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, f1_score
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, LSTM, GRU, Conv1D, GlobalAveragePooling1D,
                                     Dense, Dropout, Concatenate)
from tensorflow.keras.callbacks import EarlyStopping

# =========================
# Configuration
# =========================
ROOT_DIR = 'All_10person_Cycles'
TARGET_LEN = 100
SAMPLING_RATE = 100.0
VALID_LABELS = ['back', 'front', 'normal', 'side']

USE_FEATURES = False        # <-- toggle for ablation

IMU_COLUMNS = [
    'IMU101_v0', 'IMU101_v1', 'IMU103_v0', 'IMU103_v1',
    'IMU104_v0', 'IMU104_v1', 'IMU301_v0', 'IMU301_v1',
]
PRESSURE_COLUMNS = [
    'x0', 'x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7',
    'x8', 'x9', 'x10', 'x11', 'x12', 'x13', 'x14', 'x15'
]
CONFIGS = {
    'IMU':          IMU_COLUMNS,
    'Pressure':     PRESSURE_COLUMNS,
    'IMU+Pressure': IMU_COLUMNS + PRESSURE_COLUMNS,
}

UNITS = 64
DROPOUT_RATE = 0.3
EPOCHS = 25
BATCH_SIZE = 32
MODEL_TYPES = ['lstm', 'gru', 'cnn']

RESULTS_DIR = 'gait_hardware_paper_results'
tag = 'featconcat' if USE_FEATURES else 'seqonly_concat'
PERSUBJECT_CSV = os.path.join(RESULTS_DIR, f'loso_{tag}_persubject.csv')
POOLED_CSV = os.path.join(RESULTS_DIR, f'loso_{tag}_pooled.csv')


def get_subject_id(file_path):
    return os.path.relpath(file_path, ROOT_DIR).split(os.sep)[0]


def resample_cycle(features, target_len=TARGET_LEN):
    n = features.shape[0]
    kind = 'cubic' if n >= 4 else 'linear'
    old_t = np.linspace(0, 1, n)
    new_t = np.linspace(0, 1, target_len)
    return interp1d(old_t, features, axis=0, kind=kind)(new_t)


def load_all_raw():
    raw = []
    if not os.path.exists(ROOT_DIR):
        print(f"ROOT_DIR not found: {ROOT_DIR}")
        return raw
    for root, dirs, files in os.walk(ROOT_DIR):
        if root.endswith('Abnormal'):
            for filename in files:
                if not filename.endswith('.csv'):
                    continue
                fp = os.path.join(root, filename)
                parts = filename.replace('.csv', '').split('__')
                if len(parts) != 2:
                    continue
                name_label_part, raw_id_part = parts
                label = name_label_part.split('_')[-1]
                if raw_id_part.count('_') > 1:
                    continue
                if raw_id_part.count('_') == 1 and not raw_id_part.startswith('cycle_'):
                    continue
                cid = raw_id_part.split('_')[-1]
                if label not in VALID_LABELS or not cid.isdigit():
                    continue
                try:
                    df = pd.read_csv(fp)
                except Exception:
                    continue
                raw.append((df, label, get_subject_id(fp)))
    return raw


def build_config_matrix(raw, feature_cols):
    X, y, subj, nrows = [], [], [], []
    for df, label, s in raw:
        if any(c not in df.columns for c in feature_cols):
            continue
        vals = df[feature_cols].values
        if vals.shape[0] < 2 or vals.shape[1] == 0:
            continue
        nrows.append(vals.shape[0])
        X.append(resample_cycle(vals))
        y.append(label)
        subj.append(s)
    if len(X) == 0:
        return None, None, None, None
    return np.array(X), np.array(y), np.array(subj), np.array(nrows)


def build_model(model_type, seq_len, num_features, num_scalars, num_classes, use_features):
    seq_in = Input(shape=(seq_len, num_features), name='sequence')
    if model_type == 'lstm':
        x = LSTM(UNITS, name='LSTM')(seq_in)
        x = Dropout(DROPOUT_RATE)(x)
    elif model_type == 'gru':
        x = GRU(UNITS, name='GRU')(seq_in)
        x = Dropout(DROPOUT_RATE)(x)
    elif model_type == 'cnn':
        x = Conv1D(64, 5, activation='relu')(seq_in)
        x = Dropout(DROPOUT_RATE)(x)
        x = Conv1D(64, 5, activation='relu')(x)
        x = GlobalAveragePooling1D()(x)
        x = Dropout(DROPOUT_RATE)(x)
    else:
        raise ValueError(model_type)

    if use_features:
        scal_in = Input(shape=(num_scalars,), name='scalars')
        x = Concatenate()([x, scal_in])
        out = Dense(num_classes, activation='softmax')(x)
        model = Model(inputs=[seq_in, scal_in], outputs=out)
    else:
        out = Dense(num_classes, activation='softmax')(x)
        model = Model(inputs=seq_in, outputs=out)

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model


def run():
    raw = load_all_raw()
    if not raw:
        print("No cycles loaded.")
        return
    all_subjects = sorted({r[2] for r in raw})
    print(f"Detected {len(all_subjects)} subjects: {all_subjects}")
    print(f"USE_FEATURES = {USE_FEATURES}\n")
    if len(all_subjects) < 2:
        print("Need >= 2 subjects for LOSO. Aborting.")
        return

    global_le = LabelEncoder().fit(VALID_LABELS)
    num_classes = len(global_le.classes_)

    persubject_rows, pooled_rows = [], []

    for cfg_name, cols in CONFIGS.items():
        X, y_str, subjects, nrows = build_config_matrix(raw, cols)
        if X is None:
            print(f"[{cfg_name}] no usable cycles, skipping config")
            continue
        y_int = global_le.transform(y_str)
        y_cat = to_categorical(y_int, num_classes=num_classes)

        cadence = SAMPLING_RATE / nrows
        dur_std_by_subj = {s: float(np.std(nrows[subjects == s])) for s in np.unique(subjects)}
        dur_std = np.array([dur_std_by_subj[s] for s in subjects])
        scalar_all = np.column_stack([cadence, dur_std])
        num_scalars = scalar_all.shape[1]

        seq_len = X.shape[1]
        num_features = X.shape[2]
        unique_subjects = np.unique(subjects)
        print(f"\n{'#'*60}\n### CONFIG: {cfg_name} (n={X.shape[0]}, features={num_features})\n{'#'*60}")

        for mtype in MODEL_TYPES:
            oof_pred = np.full(len(y_int), -1, dtype=int)
            print(f"\n=== {cfg_name} | {mtype.upper()} (LOSO) ===")

            for subj in unique_subjects:
                te = (subjects == subj)
                tr = ~te
                if te.sum() == 0 or tr.sum() == 0:
                    print(f"  [{subj}] skipped (empty fold)")
                    continue

                if USE_FEATURES:
                    scaler = StandardScaler().fit(scalar_all[tr])
                    scalar_s = scaler.transform(scalar_all)
                else:
                    scalar_s = scalar_all  # unused

                tf.keras.backend.clear_session()
                model = build_model(mtype, seq_len, num_features, num_scalars,
                                    num_classes, USE_FEATURES)
                train_x = [X[tr], scalar_s[tr]] if USE_FEATURES else X[tr]
                test_x = [X[te], scalar_s[te]] if USE_FEATURES else X[te]
                model.fit(train_x, y_cat[tr], epochs=EPOCHS, batch_size=BATCH_SIZE,
                          callbacks=[EarlyStopping(monitor='loss', patience=5,
                                                   restore_best_weights=True, verbose=0)],
                          verbose=0)
                fold_pred = np.argmax(model.predict(test_x, verbose=0), axis=1)
                oof_pred[te] = fold_pred
                fold_acc = accuracy_score(y_int[te], fold_pred)
                print(f"  [{subj:15s}] n={te.sum():4d}  acc={fold_acc:.4f}")
                persubject_rows.append({
                    'config': cfg_name, 'model': mtype, 'subject': subj,
                    'use_features': USE_FEATURES, 'feature_method': 'concat',
                    'n_test': int(te.sum()), 'accuracy': round(fold_acc, 4),
                    'macro_f1': round(f1_score(y_int[te], fold_pred, average='macro', zero_division=0), 4),
                })

            done = oof_pred >= 0
            pooled_acc = accuracy_score(y_int[done], oof_pred[done])
            fa = [r['accuracy'] for r in persubject_rows
                  if r['config'] == cfg_name and r['model'] == mtype]
            mean_s = float(np.mean(fa)) if fa else float('nan')
            std_s = float(np.std(fa)) if fa else float('nan')
            worst = float(np.min(fa)) if fa else float('nan')
            print(f"  --> pooled={pooled_acc:.4f} | mean_subj={mean_s:.4f} "
                  f"(std {std_s:.4f}, worst {worst:.4f})")
            pooled_rows.append({
                'config': cfg_name, 'model': mtype,
                'use_features': USE_FEATURES, 'feature_method': 'concat',
                'pooled_accuracy': round(pooled_acc, 4),
                'mean_over_subjects': round(mean_s, 4),
                'std_over_subjects': round(std_s, 4),
                'worst_subject_acc': round(worst, 4),
                'pooled_macro_f1': round(f1_score(y_int[done], oof_pred[done],
                                                  average='macro', zero_division=0), 4),
            })

    if not pooled_rows:
        print("\nNo results produced.")
        return
    os.makedirs(RESULTS_DIR, exist_ok=True)
    pd.DataFrame(persubject_rows).to_csv(PERSUBJECT_CSV, index=False)
    pooled_df = pd.DataFrame(pooled_rows)
    pooled_df.to_csv(POOLED_CSV, index=False)
    print(f"\n\nWrote {len(persubject_rows)} per-subject rows to {PERSUBJECT_CSV}")
    print(f"Wrote {len(pooled_rows)} pooled rows to {POOLED_CSV}")
    with pd.option_context('display.width', 120):
        print(pooled_df.pivot_table(index='config', columns='model', values='pooled_accuracy'))


if __name__ == '__main__':
    tf.get_logger().setLevel('ERROR')
    run()

Detected 10 subjects: ['Andy_Dynamic', 'Ankan_Dynamic', 'David_Dynamic', 'Hrithik_Dynamic', 'JJ_Dynamic', 'Mustafa_Dynamic', 'Rezoan_Dynamic', 'Sudipta_Dynamic', 'Tianjun_Dynamic', 'Zongwei_Dynamic']
USE_FEATURES = False


############################################################
### CONFIG: IMU (n=5166, features=8)
############################################################

=== IMU | LSTM (LOSO) ===
